# Modeling: 
# Feature Selection, Tuning, and Evaluation for all four architectures

This notebook uses the per-horizon Parquet files created in `2_feature_engineering.ipynb` and performs feature selection, hyperparameter tuning, and evaluation for all four models: XGBoost, MLP, LSTM, and GNN. It also includes a topology ablation study for the GNN to assess the effect of incorporating the railway network structure.

In [66]:
import os
import json
import polars as pl
import pandas as pd
import numpy as np
import xgboost as xgb
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.metrics import *
from sklearn.preprocessing import StandardScaler

FEATURES_DIR = "/kaggle/input/notebooks/ranjithpanicker/2-feature-engineering/features"
OUTPUT_DIR = "/kaggle/working"
N_FUTURE = 10
RANDOM_STATE = 42
HORIZON_BANDS = {"near": 2, "mid": 5, "far": 8}
TUNING_HORIZONS = [2, 5, 8]

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Feature groups and feature selection configs

Defined the seven candidate feature groups every architecture chooses from for feature selection.


In [2]:
ALL_CANDIDATE_GROUPS = ["weather",
                        "infra_numeric",
                        "elektrifizierung",
                        "timetable",
                        "congestion_ahead",
                        "cyclical_extended",
                        "past_minutes_ago"]

FEATURE_SELECTION_TOLERANCE_RELATIVE = 0.005
VAL_FRACTION = 0.1
EMBED_DIM = 8
BATCH_SIZE = 4096

## Helper functions

`temporal_split_indices` is used across every model that needs a validation split.

`resolve_columns` and `resolve_exclude_groups` capture the pattern (start from a base list, adds group columns unless excluded; merges a global exclusion list with a per-horizon override from feature selection) that every model repeats with its own data. 

`run_greedy_selection` is the full backward-elimination search loop.


In [3]:
def temporal_split_indices(snapshot_times, val_fraction=VAL_FRACTION):
    order = np.argsort(snapshot_times)
    n_val = max(1, int(len(order) * val_fraction))
    return order[:-n_val], order[-n_val:]

def resolve_columns(base_columns, group_columns, exclude_groups):
    cols = list(base_columns)
    for group, cols_for_group in group_columns.items():
        if group not in exclude_groups:
            cols += cols_for_group
    return cols

def resolve_exclude_groups(base_exclude, overrides, horizon_i):
    return list(dict.fromkeys(base_exclude + overrides.get(horizon_i, [])))

def run_greedy_selection(evaluate_fn, metric_label="val MAE", horizon_bands=HORIZON_BANDS,
                          all_candidate_groups=ALL_CANDIDATE_GROUPS, tolerance=FEATURE_SELECTION_TOLERANCE_RELATIVE):
    for band_name, horizon_i in horizon_bands.items():
        excluded, remaining = [], list(all_candidate_groups)
        current_best = evaluate_fn(horizon_i, [])
        print(f"\n--- band {band_name} (horizon {horizon_i}): baseline {metric_label}={current_best:.4f} ---")
        while remaining:
            candidate_results = [(g, evaluate_fn(horizon_i, excluded + [g])) for g in remaining]
            best_group, best_val = min(candidate_results, key=lambda t: t[1])
            if best_val <= current_best * (1 + tolerance):
                excluded.append(best_group)
                remaining.remove(best_group)
                current_best = min(current_best, best_val)
                print(f"  remove '{best_group}': {metric_label}={best_val:.4f} (within tolerance, removed)")
            else:
                print(f"  stop: best remaining candidate ('{best_group}', {metric_label}={best_val:.4f}) would meaningfully hurt performance")
                break
        kept = [g for g in all_candidate_groups if g not in excluded]
        print(f"  final for {band_name}: kept={kept}, excluded={excluded}")

def save_model_config(model_name, config):
    path = f"{OUTPUT_DIR}/{model_name}_config.json"
    with open(path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"Saved confirmed configuration to {path}")

# XGBoost

## Feature-group to column mapping

Maps each of the seven candidate feature groups to the actual XGBoost input columns they cover, so a group can be included or excluded as a single unit during selection.


In [4]:
XGB_GROUP_COLUMNS = {
    "weather": ["temperature_2m", "precipitation", "wind_speed_10m"],
    "infra_numeric": ["gleisanzahl", "geschwindigkeit"],
    "timetable": ["station_headway_min", "station_dwell_min", "station_freq_per_day"],
    "congestion_ahead": ["station_congestion_n", "station_congestion_avgdelay"],
    "elektrifizierung": ["elektrifizierung_enc"],
    "past_minutes_ago": ["past_minutes_ago_1", "past_minutes_ago_2", "past_minutes_ago_3"],
    "cyclical_extended": ["hour_sin_2", "hour_sin_4", "hour_cos_2", "hour_cos_4",
                          "doy_sin_1", "doy_sin_2", "doy_sin_4", "doy_cos_1", "doy_cos_2", "doy_cos_4"]
                         + [f"dow_onehot_{d}" for d in range(7)],
}
XGB_BASE_COLUMNS = ["last_known_delay", "past_delay_1", "past_delay_2", "past_delay_3",
                    "hour_sin_1", "hour_cos_1", "dow_sin", "dow_cos",
                    "n_trains_at_current_station", "avg_delay_others_at_current_station",
                    "train_type_enc", "minutes_ahead", "station_avg_delay"]

def xgb_feature_columns(exclude_groups):
    return resolve_columns(XGB_BASE_COLUMNS, XGB_GROUP_COLUMNS, exclude_groups)

## Feature selection: greedy backward elimination

Runs the greedy backward elimination from the Helper functions section against this model's own validation loop, at the near(+2) / mid(+5) / far(+8) representative horizons.


In [11]:
XGB_FIXED_PARAMS = {"max_depth": 6, "learning_rate": 0.1}
XGB_FIXED_N_ESTIMATORS = 200

def xgb_evaluate_feature_config(horizon_i, exclude_groups):
    train_df = pl.read_parquet(f"{FEATURES_DIR}/train_h{horizon_i}.parquet")
    feature_cols = xgb_feature_columns(exclude_groups)
    fit_idx, val_idx = temporal_split_indices(train_df["snapshot_time"].to_numpy())
    X_fit, y_fit = train_df[fit_idx].select(feature_cols).to_numpy(), train_df[fit_idx]["target"].to_numpy()
    X_val, y_val = train_df[val_idx].select(feature_cols).to_numpy(), train_df[val_idx]["target"].to_numpy()
    model = xgb.XGBRegressor(n_estimators=XGB_FIXED_N_ESTIMATORS, objective="reg:squarederror",
                              n_jobs=-1, random_state=RANDOM_STATE, **XGB_FIXED_PARAMS)
    model.fit(X_fit, y_fit)
    return mean_absolute_error(y_val, model.predict(X_val))

print(f"XGBoost feature selection (candidate groups: {ALL_CANDIDATE_GROUPS})...")
run_greedy_selection(xgb_evaluate_feature_config)
print("\nFeature selection complete.")

XGBoost feature selection (candidate groups: ['weather', 'infra_numeric', 'elektrifizierung', 'timetable', 'congestion_ahead', 'cyclical_extended', 'past_minutes_ago'])...

--- band near (horizon 2): baseline val MAE=2.6805 ---
  remove 'cyclical_extended': val MAE=2.2245 (within tolerance, removed)
  remove 'weather': val MAE=2.1327 (within tolerance, removed)
  remove 'timetable': val MAE=2.0897 (within tolerance, removed)
  remove 'congestion_ahead': val MAE=2.0641 (within tolerance, removed)
  stop: best remaining candidate ('elektrifizierung', val MAE=2.0756) would meaningfully hurt performance
  final for near: kept=['infra_numeric', 'elektrifizierung', 'past_minutes_ago'], excluded=['cyclical_extended', 'weather', 'timetable', 'congestion_ahead']

--- band mid (horizon 5): baseline val MAE=0.7492 ---
  remove 'cyclical_extended': val MAE=0.6886 (within tolerance, removed)
  remove 'weather': val MAE=0.6864 (within tolerance, removed)
  remove 'congestion_ahead': val MAE=0.6895 (

## Feature Selection Config

The result of feature selection is saved here for the next stages, which includes tuning the model and final training.

`model_EXCLUDE_FEATURE_GROUPS` stores the features that gets excluded for all the horizon models (10 models).

`model_PERMANENT_HORIZON_FEATURE_OVERRIDES` stores the features that are corresponding to horizon level feature exclusion (single category of model i.e near/medium/far), these features are excluded only for the specified horizon, they are available for other horizons to train on.

In [5]:
XGB_EXCLUDE_FEATURE_GROUPS = ["cyclical_extended", "weather", "congestion_ahead", "timetable"]
XGB_PERMANENT_HORIZON_FEATURE_OVERRIDES = {}

## Hyperparameter tuning: Optuna

Pooled across horizons 2 (near), 5 (mid), and 8 (far), with early stopping on the validation split. The tuning is done with the features that were shortlisted after doing feature selection, the feature set can vary for each horizon depending on the result from the feature selection.

In [64]:
XGB_N_TUNING_TRIALS = 25

print(f"XGBoost hyperparameter tuning ({XGB_N_TUNING_TRIALS} trials, pooled across horizons {TUNING_HORIZONS})...")

xgb_tuning_data = {}
for h in TUNING_HORIZONS:
    train_df = pl.read_parquet(f"{FEATURES_DIR}/train_h{h}.parquet")
    exclude_groups = resolve_exclude_groups(XGB_EXCLUDE_FEATURE_GROUPS, XGB_PERMANENT_HORIZON_FEATURE_OVERRIDES, h)
    feature_cols = xgb_feature_columns(exclude_groups)
    fit_idx, val_idx = temporal_split_indices(train_df["snapshot_time"].to_numpy())
    xgb_tuning_data[h] = (train_df[fit_idx].select(feature_cols).to_numpy(), train_df[fit_idx]["target"].to_numpy(),
                          train_df[val_idx].select(feature_cols).to_numpy(), train_df[val_idx]["target"].to_numpy())

def xgb_objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
    }
    val_maes, best_iters = [], []
    for X_fit, y_fit, X_val, y_val in xgb_tuning_data.values():
        model = xgb.XGBRegressor(n_estimators=1000, objective="reg:squarederror", n_jobs=-1,
                                  random_state=RANDOM_STATE, early_stopping_rounds=30, eval_metric="mae", **params)
        model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
        val_maes.append(mean_absolute_error(y_val, model.predict(X_val)))
        best_iters.append(model.best_iteration if model.best_iteration is not None else 200)
    trial.set_user_attr("mean_best_iteration", float(np.mean(best_iters)))
    return float(np.mean(val_maes))

xgb_study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=5))
xgb_study.optimize(xgb_objective, n_trials=XGB_N_TUNING_TRIALS, show_progress_bar=False)
print(f"Best trial: val_MAE={xgb_study.best_value:.3f}")
print(f"Best params: {xgb_study.best_params}")
print(f"n_estimators: "
      f"{int(round(xgb_study.best_trial.user_attrs['mean_best_iteration']))}")

XGBoost hyperparameter tuning (25 trials, pooled across horizons [2, 5, 8])...


[I 2026-09-23 17:56:11,338] A new study created in memory with name: no-name-e222be3b-e5ef-449f-b8f1-edc36c3d3c82
[I 2026-09-23 17:57:50,285] Trial 0 finished with value: 0.9668353895346323 and parameters: {'max_depth': 5, 'learning_rate': 0.2536999076681772, 'subsample': 0.892797576724562, 'colsample_bytree': 0.8394633936788146, 'min_child_weight': 2, 'reg_lambda': 0.2051110418843398}. Best is trial 0 with value: 0.9668353895346323.
[I 2026-09-23 17:59:06,566] Trial 1 finished with value: 1.2043903470039368 and parameters: {'max_depth': 3, 'learning_rate': 0.19030368381735815, 'subsample': 0.8404460046972835, 'colsample_bytree': 0.8832290311184181, 'min_child_weight': 1, 'reg_lambda': 8.706020878304859}. Best is trial 0 with value: 0.9668353895346323.
[I 2026-09-23 18:01:59,895] Trial 2 finished with value: 0.9718735814094543 and parameters: {'max_depth': 9, 'learning_rate': 0.020589728197687916, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.6733618039413735, 'min_child_weigh

Best trial: val_MAE=0.778
Best params: {'max_depth': 10, 'learning_rate': 0.08882803826611764, 'subsample': 0.9337585570065023,'colsample_bytree': 0.9270610562216669, 'min_child_weight': 7, 'reg_lambda': 6.320887898989789}
n_estimators: 458
Hyperparameter tuning complete.


## Confirmed configuration

Fixed to the result of the search above, saved to a JSON file.

In [13]:
XGB_PRODUCTION_PARAMS = {
    "max_depth": 10, "learning_rate": 0.08882803826611764, "subsample": 0.9337585570065023,
    "colsample_bytree": 0.9270610562216669, "min_child_weight": 7, "reg_lambda": 6.320887898989789}

XGB_PRODUCTION_N_ESTIMATORS = 458

save_model_config("xgboost", {
    "exclude_feature_groups": XGB_EXCLUDE_FEATURE_GROUPS,
    "permanent_horizon_feature_overrides": {str(k): v for k, v in XGB_PERMANENT_HORIZON_FEATURE_OVERRIDES.items()},
    "production_params": XGB_PRODUCTION_PARAMS,
    "production_n_estimators": XGB_PRODUCTION_N_ESTIMATORS,
})

Saved confirmed configuration to /kaggle/working/xgboost_config.json


## Train and evaluate every horizon

Trains one model per horizon (1 to 10) on the full training data, using the confirmed configuration above, and evaluates each once on the held-out test set.


In [14]:
print("Training one XGBoost regressor per horizon...")

xgb_prediction_rows, xgb_all_y_true, xgb_all_pred, xgb_all_last_known = [], [], [], []
metrics_rows = []

for horizon_i in range(1, N_FUTURE + 1):
    train_df = pl.read_parquet(f"{FEATURES_DIR}/train_h{horizon_i}.parquet")
    test_df = pl.read_parquet(f"{FEATURES_DIR}/test_h{horizon_i}.parquet")
    exclude_groups = resolve_exclude_groups(XGB_EXCLUDE_FEATURE_GROUPS, XGB_PERMANENT_HORIZON_FEATURE_OVERRIDES, horizon_i)
    feature_cols = xgb_feature_columns(exclude_groups)

    model = xgb.XGBRegressor(n_estimators=XGB_PRODUCTION_N_ESTIMATORS, objective="reg:squarederror",
                              n_jobs=-1, random_state=RANDOM_STATE, **XGB_PRODUCTION_PARAMS)
    model.fit(train_df.select(feature_cols).to_numpy(), train_df["target"].to_numpy())
    
    if horizon_i in TUNING_HORIZONS:
        model.save_model(f"{OUTPUT_DIR}/xgboost_confirmed_h{horizon_i}.json")
        
    pred = model.predict(test_df.select(feature_cols).to_numpy())
    y_test, last_known = test_df["target"].to_numpy(), test_df["last_known_delay"].to_numpy()


    xgb_mae, xgb_rmse, xgb_r2 = mean_absolute_error(y_test, pred), np.sqrt(mean_squared_error(y_test, pred)), r2_score(y_test, pred)
    trans_mae, trans_rmse, trans_r2 = mean_absolute_error(y_test, last_known), np.sqrt(mean_squared_error(y_test, last_known)), r2_score(y_test, last_known)
    
    xgb_prediction_rows.append(pl.DataFrame({"horizon": horizon_i, "y_true": y_test, "pred": pred, "last_known": last_known}))
    xgb_all_y_true.append(y_test); xgb_all_pred.append(pred); xgb_all_last_known.append(last_known)
    
    print(f"  horizon {horizon_i}/10: XGBoost MAE={xgb_mae:.3f} RMSE={xgb_rmse:.3f} R2={xgb_r2:.3f}  "
          f"|  Translation MAE={trans_mae:.3f} RMSE={trans_rmse:.3f} R2={trans_r2:.3f}")

    metrics_rows.append({
    "horizon": horizon_i,
    "xgboost": xgb_mae,
    "translation": trans_mae})

pl.concat(xgb_prediction_rows).write_parquet(f"{OUTPUT_DIR}/predictions_xgboost.parquet")
print("Training complete.")

xgb_all_y_true, xgb_all_pred, xgb_all_last_known = np.concatenate(xgb_all_y_true), np.concatenate(xgb_all_pred), np.concatenate(xgb_all_last_known)

print(f"XGBoost pooled:     MAE={mean_absolute_error(xgb_all_y_true, xgb_all_pred):.3f} "
      f"RMSE={np.sqrt(mean_squared_error(xgb_all_y_true, xgb_all_pred)):.3f} R2={r2_score(xgb_all_y_true, xgb_all_pred):.3f} (n={len(xgb_all_y_true):,})")

print(f"Translation pooled: MAE={mean_absolute_error(xgb_all_y_true, xgb_all_last_known):.3f} "
      f"RMSE={np.sqrt(mean_squared_error(xgb_all_y_true, xgb_all_last_known)):.3f} R2={r2_score(xgb_all_y_true, xgb_all_last_known):.3f}")

Training one XGBoost regressor per horizon...
  horizon 1/10: XGBoost MAE=3.809 RMSE=12.501 R2=0.460  |  Translation MAE=3.844 RMSE=13.734 R2=0.348
  horizon 2/10: XGBoost MAE=3.242 RMSE=11.569 R2=0.432  |  Translation MAE=3.656 RMSE=12.948 R2=0.289
  horizon 3/10: XGBoost MAE=1.982 RMSE=9.031 R2=0.435  |  Translation MAE=2.577 RMSE=10.146 R2=0.286
  horizon 4/10: XGBoost MAE=0.864 RMSE=4.550 R2=0.612  |  Translation MAE=1.511 RMSE=6.175 R2=0.285
  horizon 5/10: XGBoost MAE=0.630 RMSE=2.995 R2=0.717  |  Translation MAE=1.277 RMSE=4.361 R2=0.400
  horizon 6/10: XGBoost MAE=0.473 RMSE=1.620 R2=0.888  |  Translation MAE=1.204 RMSE=3.424 R2=0.501
  horizon 7/10: XGBoost MAE=0.451 RMSE=1.362 R2=0.914  |  Translation MAE=1.249 RMSE=3.438 R2=0.455
  horizon 8/10: XGBoost MAE=0.436 RMSE=1.127 R2=0.939  |  Translation MAE=1.294 RMSE=3.428 R2=0.434
  horizon 9/10: XGBoost MAE=0.453 RMSE=1.506 R2=0.892  |  Translation MAE=1.347 RMSE=3.529 R2=0.409
  horizon 10/10: XGBoost MAE=0.455 RMSE=1.905 R2=

# MLP

## Feature-group to column mapping, model, and `train_one_model`

`train_type` and `elektrifizierung` are excluded from the numeric column list entirely, they go through their own embedding layer instead. The target is the residual (actual delay minus last known delay).

In [15]:
MLP_GROUP_COLUMNS = {
    "weather": ["temperature_2m", "precipitation", "wind_speed_10m"],
    "infra_numeric": ["gleisanzahl", "geschwindigkeit"],
    "timetable": ["station_headway_min", "station_dwell_min", "station_freq_per_day"],
    "congestion_ahead": ["station_congestion_n", "station_congestion_avgdelay"],
    "past_minutes_ago": ["past_minutes_ago_1", "past_minutes_ago_2", "past_minutes_ago_3"],
    "cyclical_extended": ["hour_sin_2", "hour_sin_4", "hour_cos_2", "hour_cos_4",
                          "doy_sin_1", "doy_sin_2", "doy_sin_4", "doy_cos_1", "doy_cos_2", "doy_cos_4"]
                         + [f"dow_onehot_{d}" for d in range(7)],
}
MLP_BASE_COLUMNS = ["last_known_delay", "past_delay_1", "past_delay_2", "past_delay_3",
                    "hour_sin_1", "hour_cos_1", "dow_sin", "dow_cos",
                    "n_trains_at_current_station", "avg_delay_others_at_current_station",
                    "minutes_ahead", "station_avg_delay"]

def mlp_numeric_columns(exclude_groups):
    return resolve_columns(MLP_BASE_COLUMNS, MLP_GROUP_COLUMNS, exclude_groups)

class DelayMLP(nn.Module):
    def __init__(self, n_numeric, n_train_types, n_elektrifizierung, hidden_size, n_layers, dropout):
        super().__init__()
        self.train_type_embed = nn.Embedding(n_train_types, EMBED_DIM)
        self.has_elektrifizierung = n_elektrifizierung > 0
        self.elektrifizierung_embed = nn.Embedding(max(n_elektrifizierung, 1), EMBED_DIM)
        input_dim = n_numeric + EMBED_DIM + (EMBED_DIM if self.has_elektrifizierung else 0)
        layers = []
        dim = input_dim
        for _ in range(n_layers):
            layers += [nn.Linear(dim, hidden_size), nn.ReLU(), nn.Dropout(dropout)]
            dim = hidden_size
        layers.append(nn.Linear(dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, numeric, train_type_ids, elektrifizierung_ids=None):
        tt_embed = self.train_type_embed(train_type_ids)
        parts = [numeric, tt_embed]
        if self.has_elektrifizierung and elektrifizierung_ids is not None:
            parts.append(self.elektrifizierung_embed(elektrifizierung_ids))
        return self.net(torch.cat(parts, dim=1)).squeeze(-1)

def mlp_batched_forward(model, batch_size, numeric, tt, elek=None):
    preds = []
    n = numeric.shape[0]
    for start in range(0, n, batch_size):
        end = start + batch_size
        preds.append(model(numeric[start:end], tt[start:end], elek[start:end] if elek is not None else None))
    return torch.cat(preds, dim=0)

def mlp_train_one_model(train_df, numeric_cols, has_elek, n_train_types, n_elektrifizierung,
                         hidden_size, n_layers, dropout, learning_rate, max_epochs, patience):
    fit_idx, val_idx = temporal_split_indices(train_df["snapshot_time"].to_numpy())
    fit_df, val_df = train_df[fit_idx], train_df[val_idx]
    scaler = StandardScaler().fit(fit_df.select(numeric_cols).to_numpy())

    def make_tensors(df):
        numeric = torch.tensor(scaler.transform(df.select(numeric_cols).to_numpy()), dtype=torch.float32, device=DEVICE)
        tt = torch.tensor(df["train_type_enc"].to_numpy(), dtype=torch.long, device=DEVICE)
        elek = torch.tensor(df["elektrifizierung_enc"].to_numpy(), dtype=torch.long, device=DEVICE) if has_elek else None
        y = torch.tensor((df["target"] - df["last_known_delay"]).to_numpy(), dtype=torch.float32, device=DEVICE)
        return numeric, tt, elek, y

    numeric_fit, tt_fit, elek_fit, y_fit = make_tensors(fit_df)
    numeric_val, tt_val, elek_val, y_val = make_tensors(val_df)

    torch.manual_seed(RANDOM_STATE)
    model = DelayMLP(len(numeric_cols), n_train_types, n_elektrifizierung, hidden_size, n_layers, dropout).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=LR_SCHEDULER_FACTOR,
                                                            patience=LR_SCHEDULER_PATIENCE)
    loss_fn = nn.L1Loss()
    n_fit = numeric_fit.shape[0]
    best_val_mae, best_state, epochs_without_improve = float("inf"), None, 0
    for epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(n_fit, device=DEVICE)
        for start in range(0, n_fit, BATCH_SIZE):
            batch_idx = perm[start:start + BATCH_SIZE]
            optimizer.zero_grad()
            preds = model(numeric_fit[batch_idx], tt_fit[batch_idx], elek_fit[batch_idx] if elek_fit is not None else None)
            loss = loss_fn(preds, y_fit[batch_idx])
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_mae = mean_absolute_error(y_val.cpu().numpy(),
                                          mlp_batched_forward(model, BATCH_SIZE, numeric_val, tt_val, elek_val).cpu().numpy())
        scheduler.step(val_mae)
        if val_mae < best_val_mae - 1e-4:
            best_val_mae, epochs_without_improve, best_state = val_mae, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    del numeric_fit, tt_fit, elek_fit, y_fit, numeric_val, tt_val, elek_val, y_val
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return model, best_val_mae, scaler

## Feature selection: greedy backward elimination

Runs the greedy backward elimination from the Helper functions section against this model's own validation loop, at the near(+2) / mid(+5) / far(+8) representative horizons.


In [21]:
MLP_FIXED_HP = {"hidden_size": 64, "n_layers": 2, "dropout": 0.1, "learning_rate": 1e-2}
MLP_SELECTION_MAX_EPOCHS, MLP_SELECTION_PATIENCE = 20, 6
LR_SCHEDULER_FACTOR, LR_SCHEDULER_PATIENCE = 0.5, 4

def mlp_evaluate_feature_config(horizon_i, exclude_groups):
    train_df = pl.read_parquet(f"{FEATURES_DIR}/train_h{horizon_i}.parquet")
    numeric_cols = mlp_numeric_columns(exclude_groups)
    has_elek = "elektrifizierung" not in exclude_groups
    n_train_types = train_df["train_type_enc"].max() + 1
    n_elektrifizierung = (train_df["elektrifizierung_enc"].max() + 1) if has_elek else 0
    _, val_mae, _ = mlp_train_one_model(train_df, numeric_cols, has_elek, n_train_types, n_elektrifizierung,
                                        MLP_FIXED_HP["hidden_size"], MLP_FIXED_HP["n_layers"], MLP_FIXED_HP["dropout"],
                                        MLP_FIXED_HP["learning_rate"], MLP_SELECTION_MAX_EPOCHS, MLP_SELECTION_PATIENCE)
    return val_mae

print(f"MLP feature selection (candidate groups: {ALL_CANDIDATE_GROUPS})...")
run_greedy_selection(mlp_evaluate_feature_config, metric_label="val residual MAE")
print("\nFeature selection complete.")

MLP feature selection (candidate groups: ['weather', 'infra_numeric', 'elektrifizierung', 'timetable', 'congestion_ahead', 'cyclical_extended', 'past_minutes_ago'])...

--- band near (horizon 2): baseline val residual MAE=2.3222 ---
  remove 'cyclical_extended': val residual MAE=1.8009 (within tolerance, removed)
  remove 'weather': val residual MAE=1.7086 (within tolerance, removed)
  remove 'congestion_ahead': val residual MAE=1.7097 (within tolerance, removed)
  remove 'elektrifizierung': val residual MAE=1.6725 (within tolerance, removed)
  stop: best remaining candidate ('timetable', val residual MAE=1.7083) would meaningfully hurt performance
  final for near: kept=['infra_numeric', 'timetable', 'past_minutes_ago'], excluded=['cyclical_extended', 'weather', 'congestion_ahead', 'elektrifizierung']

--- band mid (horizon 5): baseline val residual MAE=0.6899 ---
  remove 'cyclical_extended': val residual MAE=0.5602 (within tolerance, removed)
  remove 'weather': val residual MAE=0.5

## Feature Selection Config

The result of feature selection is saved here for the next stages, which includes tuning the model and final training.

`model_EXCLUDE_FEATURE_GROUPS` stores the features that gets excluded for all the horizon models (10 models).

`model_PERMANENT_HORIZON_FEATURE_OVERRIDES` stores the features that are corresponding to horizon level feature exclusion (single category of model i.e near/medium/far), these features are excluded only for the specified horizon, they are available for other horizons to train on.

In [16]:
MLP_EXCLUDE_FEATURE_GROUPS = ["cyclical_extended", "weather"]

MLP_PERMANENT_HORIZON_FEATURE_OVERRIDES = {
    1: ["congestion_ahead", "elektrifizierung"], 2: ["congestion_ahead", "elektrifizierung"], 3: ["congestion_ahead", "elektrifizierung"],
    4: ["congestion_ahead"], 5: ["congestion_ahead"], 6: ["congestion_ahead"], 7: ["congestion_ahead"],
}

## Hyperparameter tuning: Optuna

Pooled across horizons 2 (near), 5 (mid), and 8 (far), with early stopping on the validation split. The tuning is done with the features that were shortlisted after doing feature selection, the feature set can vary for each horizon depending on the result from the feature selection.

In [23]:
MLP_N_TUNING_TRIALS = 15
MLP_TUNING_MAX_EPOCHS, MLP_TUNING_PATIENCE = 30, 8

MLP_TUNING_EXCLUDE_FEATURE_GROUPS = ["cyclical_extended", "weather"]
MLP_TUNING_PERMANENT_HORIZON_FEATURE_OVERRIDES = {8: ["congestion_ahead"], 9: ["congestion_ahead"], 10: ["congestion_ahead"]}

print(f"MLP hyperparameter tuning ({MLP_N_TUNING_TRIALS} trials, pooled across horizons {TUNING_HORIZONS})...")
mlp_tuning_dfs = {h: pl.read_parquet(f"{FEATURES_DIR}/train_h{h}.parquet") for h in TUNING_HORIZONS}

def mlp_objective(trial):
    hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128, 256])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.0, 0.4)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 2e-2, log=True)
    val_maes = []
    for h, train_df in mlp_tuning_dfs.items():
        exclude_groups = resolve_exclude_groups(MLP_EXCLUDE_FEATURE_GROUPS, MLP_PERMANENT_HORIZON_FEATURE_OVERRIDES, h)
        numeric_cols = mlp_numeric_columns(exclude_groups)
        has_elek = "elektrifizierung" not in exclude_groups
        n_train_types = train_df["train_type_enc"].max() + 1
        n_elektrifizierung = (train_df["elektrifizierung_enc"].max() + 1) if has_elek else 0
        _, val_mae, _ = mlp_train_one_model(train_df, numeric_cols, has_elek, n_train_types, n_elektrifizierung,
                                            hidden_size, n_layers, dropout, learning_rate,
                                            MLP_TUNING_MAX_EPOCHS, MLP_TUNING_PATIENCE)
        val_maes.append(val_mae)
    return float(np.mean(val_maes))

mlp_study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=5))
mlp_study.optimize(mlp_objective, n_trials=MLP_N_TUNING_TRIALS, show_progress_bar=False)
print(f"Best trial: val_residual_MAE={mlp_study.best_value:.3f}")
print(f"Best params: {mlp_study.best_params}")
print("Hyperparameter tuning complete.")

MLP hyperparameter tuning (15 trials, pooled across horizons [2, 5, 8])...


[I 2026-09-19 16:07:06,785] A new study created in memory with name: no-name-dc06dd04-29e4-4549-884c-184f7547416f
[I 2026-09-19 16:08:09,932] Trial 0 finished with value: 1.3045463959376018 and parameters: {'hidden_size': 64, 'n_layers': 1, 'dropout': 0.062397808134481064, 'learning_rate': 0.0001360354613611809}. Best is trial 0 with value: 1.3045463959376018.
[I 2026-09-19 16:09:38,980] Trial 1 finished with value: 1.1480427781740825 and parameters: {'hidden_size': 32, 'n_layers': 3, 'dropout': 0.3329770563201687, 'learning_rate': 0.0003080340052983973}. Best is trial 1 with value: 1.1480427781740825.
[I 2026-09-19 16:10:55,933] Trial 2 finished with value: 0.7606133917967478 and parameters: {'hidden_size': 256, 'n_layers': 2, 'dropout': 0.11649165607921677, 'learning_rate': 0.002557948896094737}. Best is trial 2 with value: 0.7606133917967478.
[I 2026-09-19 16:12:26,916] Trial 3 finished with value: 0.6757330497105917 and parameters: {'hidden_size': 256, 'n_layers': 3, 'dropout': 0.0

Best trial: val_residual_MAE=0.676
Best params: {'hidden_size': 256, 'n_layers': 3, 'dropout': 0.0798695128633439, 'learning_rate': 0.001524996572521357}
Hyperparameter tuning complete.


## Confirmed configuration

The final, locked feature exclusions and hyperparameters, used to train this model's production run below.


In [17]:
MLP_PRODUCTION_HP = {"hidden_size": 256, "n_layers": 3, "dropout": 0.0798695128633439,
                      "learning_rate": 0.001524996572521357}

MLP_MAX_EPOCHS, MLP_PATIENCE = 120, 15

save_model_config("mlp", {
    "exclude_feature_groups": MLP_EXCLUDE_FEATURE_GROUPS,
    "permanent_horizon_feature_overrides": {str(k): v for k, v in MLP_PERMANENT_HORIZON_FEATURE_OVERRIDES.items()},
    "production_hp": MLP_PRODUCTION_HP,
})

Saved confirmed configuration to /kaggle/working/mlp_config.json


## Train and evaluate every horizon

Trains one model per horizon (1 to 10) on the full training data, using the confirmed configuration above, and evaluates each once on the held-out test set.


In [20]:
print("Training one MLP regressor per horizon...")
mlp_prediction_rows, mlp_all_y_true, mlp_all_pred, mlp_all_last_known = [], [], [], []
mpl_metrics_rows = []

for horizon_i in range(1, N_FUTURE + 1):
    train_df = pl.read_parquet(f"{FEATURES_DIR}/train_h{horizon_i}.parquet")
    test_df = pl.read_parquet(f"{FEATURES_DIR}/test_h{horizon_i}.parquet")
    exclude_groups = resolve_exclude_groups(MLP_EXCLUDE_FEATURE_GROUPS, MLP_PERMANENT_HORIZON_FEATURE_OVERRIDES, horizon_i)
    numeric_cols = mlp_numeric_columns(exclude_groups)
    has_elek = "elektrifizierung" not in exclude_groups
    n_train_types = train_df["train_type_enc"].max() + 1
    n_elektrifizierung = (train_df["elektrifizierung_enc"].max() + 1) if has_elek else 0

    model, best_val_mae, scaler = mlp_train_one_model(
        train_df, numeric_cols, has_elek, n_train_types, n_elektrifizierung,
        MLP_PRODUCTION_HP["hidden_size"], MLP_PRODUCTION_HP["n_layers"], MLP_PRODUCTION_HP["dropout"],
        MLP_PRODUCTION_HP["learning_rate"], MLP_MAX_EPOCHS, MLP_PATIENCE)

    numeric_test = torch.tensor(scaler.transform(test_df.select(numeric_cols).to_numpy()), dtype=torch.float32, device=DEVICE)
    tt_test = torch.tensor(test_df["train_type_enc"].to_numpy(), dtype=torch.long, device=DEVICE)
    elek_test = torch.tensor(test_df["elektrifizierung_enc"].to_numpy(), dtype=torch.long, device=DEVICE) if has_elek else None
    with torch.no_grad():
        pred_residual = mlp_batched_forward(model, BATCH_SIZE, numeric_test, tt_test, elek_test).cpu().numpy()
    last_known = test_df["last_known_delay"].to_numpy()
    y_test = test_df["target"].to_numpy()
    pred_absolute = pred_residual + last_known

    mlp_mae, mlp_rmse, mlp_r2 = mean_absolute_error(y_test, pred_absolute), np.sqrt(mean_squared_error(y_test, pred_absolute)), r2_score(y_test, pred_absolute)
    trans_mae, trans_rmse, trans_r2 = mean_absolute_error(y_test, last_known), np.sqrt(mean_squared_error(y_test, last_known)), r2_score(y_test, last_known)
    mlp_prediction_rows.append(pl.DataFrame({"horizon": horizon_i, "y_true": y_test, "pred": pred_absolute, "last_known": last_known}))
    mlp_all_y_true.append(y_test); mlp_all_pred.append(pred_absolute); mlp_all_last_known.append(last_known)
    print(f"  horizon {horizon_i}/10 (best val_residual_MAE={best_val_mae:.3f}): "
          f"MLP MAE={mlp_mae:.3f} RMSE={mlp_rmse:.3f} R2={mlp_r2:.3f}  "
          f"|  Translation MAE={trans_mae:.3f} RMSE={trans_rmse:.3f} R2={trans_r2:.3f}")

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    mpl_metrics_rows.append({
    "horizon": horizon_i,
    "MLP": mlp_mae})

pl.concat(mlp_prediction_rows).write_parquet(f"{OUTPUT_DIR}/predictions_mlp.parquet")
print("Training complete.")

mlp_all_y_true, mlp_all_pred, mlp_all_last_known = np.concatenate(mlp_all_y_true), np.concatenate(mlp_all_pred), np.concatenate(mlp_all_last_known)
print(f"MLP pooled:         MAE={mean_absolute_error(mlp_all_y_true, mlp_all_pred):.3f} "
      f"RMSE={np.sqrt(mean_squared_error(mlp_all_y_true, mlp_all_pred)):.3f} R2={r2_score(mlp_all_y_true, mlp_all_pred):.3f} (n={len(mlp_all_y_true):,})")
print(f"Translation pooled: MAE={mean_absolute_error(mlp_all_y_true, mlp_all_last_known):.3f} "
      f"RMSE={np.sqrt(mean_squared_error(mlp_all_y_true, mlp_all_last_known)):.3f} R2={r2_score(mlp_all_y_true, mlp_all_last_known):.3f}")

Training one MLP regressor per horizon...
  horizon 1/10 (best val_residual_MAE=1.737): MLP MAE=3.464 RMSE=11.748 R2=0.523  |  Translation MAE=3.844 RMSE=13.734 R2=0.348
  horizon 2/10 (best val_residual_MAE=1.276): MLP MAE=2.950 RMSE=10.663 R2=0.518  |  Translation MAE=3.656 RMSE=12.948 R2=0.289
  horizon 3/10 (best val_residual_MAE=0.901): MLP MAE=1.858 RMSE=10.304 R2=0.264  |  Translation MAE=2.577 RMSE=10.146 R2=0.286
  horizon 4/10 (best val_residual_MAE=0.567): MLP MAE=0.759 RMSE=7.766 R2=-0.131  |  Translation MAE=1.511 RMSE=6.175 R2=0.285
  horizon 5/10 (best val_residual_MAE=0.341): MLP MAE=0.446 RMSE=4.929 R2=0.233  |  Translation MAE=1.277 RMSE=4.361 R2=0.400
  horizon 6/10 (best val_residual_MAE=0.314): MLP MAE=0.353 RMSE=3.245 R2=0.552  |  Translation MAE=1.204 RMSE=3.424 R2=0.501
  horizon 7/10 (best val_residual_MAE=0.291): MLP MAE=0.330 RMSE=3.312 R2=0.494  |  Translation MAE=1.249 RMSE=3.438 R2=0.455
  horizon 8/10 (best val_residual_MAE=0.292): MLP MAE=0.336 RMSE=2.41

# LSTM

## Feature-group to column mapping, model, and `train_one_model`

The target train's own recent history (`past_delay_3`, `past_delay_2`, `past_delay_1`, `last_known_delay`) is fed through the LSTM as a length-4 sequence, not as static features; `past_minutes_ago` stays static.

In [23]:
LSTM_GROUP_COLUMNS = {
    "weather": ["temperature_2m", "precipitation", "wind_speed_10m"],
    "infra_numeric": ["gleisanzahl", "geschwindigkeit"],
    "timetable": ["station_headway_min", "station_dwell_min", "station_freq_per_day"],
    "congestion_ahead": ["station_congestion_n", "station_congestion_avgdelay"],
    "past_minutes_ago": ["past_minutes_ago_1", "past_minutes_ago_2", "past_minutes_ago_3"],
    "cyclical_extended": ["hour_sin_2", "hour_sin_4", "hour_cos_2", "hour_cos_4",
                          "doy_sin_1", "doy_sin_2", "doy_sin_4", "doy_cos_1", "doy_cos_2", "doy_cos_4"]
                         + [f"dow_onehot_{d}" for d in range(7)],
}

LSTM_STATIC_BASE_COLUMNS = ["hour_sin_1", "hour_cos_1", "dow_sin", "dow_cos",
                            "n_trains_at_current_station", "avg_delay_others_at_current_station",
                            "minutes_ahead", "station_avg_delay"]
SEQUENCE_COLS = ["past_delay_3", "past_delay_2", "past_delay_1", "last_known_delay"]
SEQ_LEN = len(SEQUENCE_COLS)

def lstm_static_columns(exclude_groups):
    return resolve_columns(LSTM_STATIC_BASE_COLUMNS, LSTM_GROUP_COLUMNS, exclude_groups)

class SeqScaler:
    def __init__(self, seq):
        self.mean = seq.mean()
        std = seq.std()
        self.std = std if std > 1e-6 else 1.0

    def transform(self, seq):
        return (seq - self.mean) / self.std

class DelayLSTM(nn.Module):
    def __init__(self, n_static_numeric, n_train_types, n_elektrifizierung, hidden_size, lstm_layers, mlp_hidden, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, num_layers=lstm_layers, batch_first=True)
        self.train_type_embed = nn.Embedding(n_train_types, EMBED_DIM)
        self.has_elektrifizierung = n_elektrifizierung > 0
        self.elektrifizierung_embed = nn.Embedding(max(n_elektrifizierung, 1), EMBED_DIM)
        combined_dim = hidden_size + n_static_numeric + EMBED_DIM + (EMBED_DIM if self.has_elektrifizierung else 0)
        self.head = nn.Sequential(
            nn.Linear(combined_dim, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, mlp_hidden // 2), nn.ReLU(), nn.Linear(mlp_hidden // 2, 1))

    def forward(self, seq, static_numeric, train_type_ids, elektrifizierung_ids=None):
        _, (h_n, _) = self.lstm(seq)
        parts = [h_n[-1], static_numeric, self.train_type_embed(train_type_ids)]
        if self.has_elektrifizierung and elektrifizierung_ids is not None:
            parts.append(self.elektrifizierung_embed(elektrifizierung_ids))
        return self.head(torch.cat(parts, dim=1)).squeeze(-1)

def lstm_batched_forward(model, batch_size, seq, static, tt, elek=None):
    preds = []
    n = seq.shape[0]
    for start in range(0, n, batch_size):
        end = start + batch_size
        preds.append(model(seq[start:end], static[start:end], tt[start:end], elek[start:end] if elek is not None else None))
    return torch.cat(preds, dim=0)

def lstm_train_one_model(train_df, static_cols, has_elek, n_train_types, n_elektrifizierung,
                          hidden_size, lstm_layers, mlp_hidden, dropout, learning_rate, max_epochs, patience):
    fit_idx, val_idx = temporal_split_indices(train_df["snapshot_time"].to_numpy())
    fit_df, val_df = train_df[fit_idx], train_df[val_idx]
    static_scaler = StandardScaler().fit(fit_df.select(static_cols).to_numpy())
    seq_scaler = SeqScaler(fit_df.select(SEQUENCE_COLS).to_numpy())

    def make_tensors(df):
        seq = seq_scaler.transform(df.select(SEQUENCE_COLS).to_numpy().reshape(-1, SEQ_LEN, 1))
        static = static_scaler.transform(df.select(static_cols).to_numpy())
        tt = df["train_type_enc"].to_numpy()
        elek = df["elektrifizierung_enc"].to_numpy() if has_elek else None
        y = (df["target"] - df["last_known_delay"]).to_numpy()
        return (torch.tensor(seq, dtype=torch.float32, device=DEVICE),
                torch.tensor(static, dtype=torch.float32, device=DEVICE),
                torch.tensor(tt, dtype=torch.long, device=DEVICE),
                torch.tensor(elek, dtype=torch.long, device=DEVICE) if has_elek else None,
                torch.tensor(y, dtype=torch.float32, device=DEVICE))

    seq_fit, static_fit, tt_fit, elek_fit, y_fit = make_tensors(fit_df)
    seq_val, static_val, tt_val, elek_val, y_val = make_tensors(val_df)

    torch.manual_seed(RANDOM_STATE)
    model = DelayLSTM(len(static_cols), n_train_types, n_elektrifizierung, hidden_size, lstm_layers, mlp_hidden, dropout).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=LR_SCHEDULER_FACTOR,
                                                            patience=LR_SCHEDULER_PATIENCE)
    loss_fn = nn.L1Loss()
    n_fit = seq_fit.shape[0]
    best_val_mae, best_state, epochs_without_improve = float("inf"), None, 0
    for epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(n_fit, device=DEVICE)
        for start in range(0, n_fit, BATCH_SIZE):
            batch_idx = perm[start:start + BATCH_SIZE]
            optimizer.zero_grad()
            preds = model(seq_fit[batch_idx], static_fit[batch_idx], tt_fit[batch_idx],
                          elek_fit[batch_idx] if elek_fit is not None else None)
            loss = loss_fn(preds, y_fit[batch_idx])
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_mae = mean_absolute_error(y_val.cpu().numpy(),
                                          lstm_batched_forward(model, BATCH_SIZE, seq_val, static_val, tt_val, elek_val).cpu().numpy())
        scheduler.step(val_mae)
        if val_mae < best_val_mae - 1e-4:
            best_val_mae, epochs_without_improve, best_state = val_mae, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    del seq_fit, static_fit, tt_fit, elek_fit, y_fit, seq_val, static_val, tt_val, elek_val, y_val
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return model, best_val_mae, static_scaler, seq_scaler

## Feature selection: greedy backward elimination

Runs the greedy backward elimination from the Helper functions section against this model's own validation loop, at the near(+2) / mid(+5) / far(+8) representative horizons.


In [4]:
LSTM_FIXED_HP = {"hidden_size": 64, "lstm_layers": 1, "mlp_hidden": 64, "dropout": 0.1, "learning_rate": 1e-2}
LSTM_SELECTION_MAX_EPOCHS, LSTM_SELECTION_PATIENCE = 15, 5

def lstm_evaluate_feature_config(horizon_i, exclude_groups):
    train_df = pl.read_parquet(f"{FEATURES_DIR}/train_h{horizon_i}.parquet")
    static_cols = lstm_static_columns(exclude_groups)
    has_elek = "elektrifizierung" not in exclude_groups
    n_train_types = train_df["train_type_enc"].max() + 1
    n_elektrifizierung = (train_df["elektrifizierung_enc"].max() + 1) if has_elek else 0
    _, val_mae, _, _ = lstm_train_one_model(train_df, static_cols, has_elek, n_train_types, n_elektrifizierung,
                                            LSTM_FIXED_HP["hidden_size"], LSTM_FIXED_HP["lstm_layers"], LSTM_FIXED_HP["mlp_hidden"],
                                            LSTM_FIXED_HP["dropout"], LSTM_FIXED_HP["learning_rate"],
                                            LSTM_SELECTION_MAX_EPOCHS, LSTM_SELECTION_PATIENCE)
    return val_mae

print(f"LSTM feature selection (candidate groups: {ALL_CANDIDATE_GROUPS})...")
run_greedy_selection(lstm_evaluate_feature_config, metric_label="val residual MAE")
print("\nFeature selection complete.")

LSTM feature selection (candidate groups: ['weather', 'infra_numeric', 'elektrifizierung', 'timetable', 'congestion_ahead', 'cyclical_extended', 'past_minutes_ago'])...

--- band near (horizon 2): baseline val residual MAE=2.2095 ---
  remove 'cyclical_extended': val residual MAE=1.8668 (within tolerance, removed)
  remove 'weather': val residual MAE=1.7615 (within tolerance, removed)
  remove 'elektrifizierung': val residual MAE=1.7123 (within tolerance, removed)
  remove 'congestion_ahead': val residual MAE=1.6777 (within tolerance, removed)
  stop: best remaining candidate ('timetable', val residual MAE=1.7138) would meaningfully hurt performance
  final for near: kept=['infra_numeric', 'timetable', 'past_minutes_ago'], excluded=['cyclical_extended', 'weather', 'elektrifizierung', 'congestion_ahead']

--- band mid (horizon 5): baseline val residual MAE=0.6687 ---
  remove 'cyclical_extended': val residual MAE=0.5654 (within tolerance, removed)
  remove 'weather': val residual MAE=0.

## Feature Selection Config

The result of feature selection is saved here for the next stages, which includes tuning the model and final training.

`model_EXCLUDE_FEATURE_GROUPS` stores the features that gets excluded for all the horizon models (10 models).

`model_PERMANENT_HORIZON_FEATURE_OVERRIDES` stores the features that are corresponding to horizon level feature exclusion (single category of model i.e near/medium/far), these features are excluded only for the specified horizon, they are available for other horizons to train on.

In [24]:
LSTM_EXCLUDE_FEATURE_GROUPS = ['cyclical_extended', 'weather']

LSTM_PERMANENT_HORIZON_FEATURE_OVERRIDES = {
    1: ['congestion_ahead', 'elektrifizierung'], 2: ['congestion_ahead', 'elektrifizierung'], 3: ['congestion_ahead', 'elektrifizierung'],
    4: ['congestion_ahead'], 5: ['congestion_ahead'], 6: ['congestion_ahead'], 7: ['congestion_ahead'],}

## Hyperparameter tuning: Optuna


In [5]:
LSTM_N_TUNING_TRIALS = 15
LSTM_TUNING_MAX_EPOCHS, LSTM_TUNING_PATIENCE = 30, 8


print(f"LSTM hyperparameter tuning ({LSTM_N_TUNING_TRIALS} trials, pooled across horizons {TUNING_HORIZONS})...")
lstm_tuning_dfs = {h: pl.read_parquet(f"{FEATURES_DIR}/train_h{h}.parquet") for h in TUNING_HORIZONS}

def lstm_objective(trial):
    hp = {
        "hidden_size": trial.suggest_categorical("hidden_size", [32, 64, 128, 256]),
        "lstm_layers": trial.suggest_int("lstm_layers", 1, 3),
        "mlp_hidden": trial.suggest_categorical("mlp_hidden", [32, 64, 128]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
    }
    val_maes = []
    for h, train_df in lstm_tuning_dfs.items():
        exclude_groups = resolve_exclude_groups(LSTM_EXCLUDE_FEATURE_GROUPS, LSTM_PERMANENT_HORIZON_FEATURE_OVERRIDES, h)
        static_cols = lstm_static_columns(exclude_groups)
        has_elek = "elektrifizierung" not in exclude_groups
        n_train_types = train_df["train_type_enc"].max() + 1
        n_elektrifizierung = (train_df["elektrifizierung_enc"].max() + 1) if has_elek else 0
        _, val_mae, _, _ = lstm_train_one_model(train_df, static_cols, has_elek, n_train_types, n_elektrifizierung,
                                                hp["hidden_size"], hp["lstm_layers"], hp["mlp_hidden"], hp["dropout"],
                                                hp["learning_rate"], LSTM_TUNING_MAX_EPOCHS, LSTM_TUNING_PATIENCE)
        val_maes.append(val_mae)
    return float(np.mean(val_maes))

lstm_study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=5))
lstm_study.optimize(lstm_objective, n_trials=LSTM_N_TUNING_TRIALS, show_progress_bar=False)
print(f"Best trial: val_residual_MAE={lstm_study.best_value:.3f}")
print(f"Best params: {lstm_study.best_params}")
print("Hyperparameter tuning complete.")

LSTM hyperparameter tuning (15 trials, pooled across horizons [2, 5, 8])...


[I 2026-09-16 06:37:37,758] A new study created in memory with name: no-name-f02e25a4-ef4c-4bd9-b12c-cb2a65f3db9e
[I 2026-09-16 06:40:21,407] Trial 0 finished with value: 0.8888347148895264 and parameters: {'hidden_size': 64, 'lstm_layers': 1, 'mlp_hidden': 128, 'dropout': 0.3005575058716044, 'learning_rate': 0.0026070247583707684}. Best is trial 0 with value: 0.8888347148895264.
[I 2026-09-16 06:43:10,499] Trial 1 finished with value: 0.9705864588419596 and parameters: {'hidden_size': 64, 'lstm_layers': 1, 'mlp_hidden': 128, 'dropout': 0.21597250932105788, 'learning_rate': 0.0003823475224675188}. Best is trial 0 with value: 0.8888347148895264.
[I 2026-09-16 06:45:07,966] Trial 2 finished with value: 1.294939895470937 and parameters: {'hidden_size': 32, 'lstm_layers': 2, 'mlp_hidden': 32, 'dropout': 0.29620728443102123, 'learning_rate': 0.0001238513729886094}. Best is trial 0 with value: 0.8888347148895264.
[I 2026-09-16 07:09:15,327] Trial 3 finished with value: 1.1780295769373577 and

Best trial: val_residual_MAE=0.819
Best params: {'hidden_size': 128, 'lstm_layers': 3, 'mlp_hidden': 128, 'dropout': 0.008980937807409828, 'learning_rate': 0.00887353789827113}
Hyperparameter tuning complete.


## Confirmed configuration

The final, locked feature exclusions and hyperparameters, used to train this model's production run below.


In [25]:
LSTM_PRODUCTION_HP = {'hidden_size': 128, 'lstm_layers': 3, 'mlp_hidden': 128, 'dropout': 0.008980937807409828, 'learning_rate': 0.00887353789827113}

LSTM_MAX_EPOCHS, LSTM_PATIENCE = 120, 15

save_model_config("lstm", {
    "exclude_feature_groups": LSTM_EXCLUDE_FEATURE_GROUPS,
    "permanent_horizon_feature_overrides": {str(k): v for k, v in LSTM_PERMANENT_HORIZON_FEATURE_OVERRIDES.items()},
    "production_hp": LSTM_PRODUCTION_HP,
})

Saved confirmed configuration to /kaggle/working/lstm_config.json


## Train and evaluate every horizon

Trains one model per horizon (1 to 10) on the full training data, using the confirmed configuration above, and evaluates each once on the held-out test set.


In [26]:
print("Training one LSTM regressor per horizon...")
lstm_prediction_rows, lstm_all_y_true, lstm_all_pred, lstm_all_last_known = [], [], [], []
lstm_metrics_rows = []

for horizon_i in range(1, N_FUTURE + 1):
    train_df = pl.read_parquet(f"{FEATURES_DIR}/train_h{horizon_i}.parquet")
    test_df = pl.read_parquet(f"{FEATURES_DIR}/test_h{horizon_i}.parquet")
    exclude_groups = resolve_exclude_groups(LSTM_EXCLUDE_FEATURE_GROUPS, LSTM_PERMANENT_HORIZON_FEATURE_OVERRIDES, horizon_i)
    static_cols = lstm_static_columns(exclude_groups)
    has_elek = "elektrifizierung" not in exclude_groups
    n_train_types = train_df["train_type_enc"].max() + 1
    n_elektrifizierung = (train_df["elektrifizierung_enc"].max() + 1) if has_elek else 0

    model, best_val_mae, static_scaler, seq_scaler = lstm_train_one_model(
        train_df, static_cols, has_elek, n_train_types, n_elektrifizierung,
        LSTM_PRODUCTION_HP["hidden_size"], LSTM_PRODUCTION_HP["lstm_layers"], LSTM_PRODUCTION_HP["mlp_hidden"],
        LSTM_PRODUCTION_HP["dropout"], LSTM_PRODUCTION_HP["learning_rate"], LSTM_MAX_EPOCHS, LSTM_PATIENCE)

    seq_test = seq_scaler.transform(test_df.select(SEQUENCE_COLS).to_numpy().reshape(-1, SEQ_LEN, 1))
    static_test = static_scaler.transform(test_df.select(static_cols).to_numpy())
    seq_test_t = torch.tensor(seq_test, dtype=torch.float32, device=DEVICE)
    static_test_t = torch.tensor(static_test, dtype=torch.float32, device=DEVICE)
    tt_test = torch.tensor(test_df["train_type_enc"].to_numpy(), dtype=torch.long, device=DEVICE)
    elek_test = torch.tensor(test_df["elektrifizierung_enc"].to_numpy(), dtype=torch.long, device=DEVICE) if has_elek else None
    with torch.no_grad():
        pred_residual = lstm_batched_forward(model, BATCH_SIZE, seq_test_t, static_test_t, tt_test, elek_test).cpu().numpy()
    last_known = test_df["last_known_delay"].to_numpy()
    y_test = test_df["target"].to_numpy()
    pred_absolute = pred_residual + last_known

    lstm_mae, lstm_rmse, lstm_r2 = mean_absolute_error(y_test, pred_absolute), np.sqrt(mean_squared_error(y_test, pred_absolute)), r2_score(y_test, pred_absolute)
    trans_mae, trans_rmse, trans_r2 = mean_absolute_error(y_test, last_known), np.sqrt(mean_squared_error(y_test, last_known)), r2_score(y_test, last_known)
    lstm_prediction_rows.append(pl.DataFrame({"horizon": horizon_i, "y_true": y_test, "pred": pred_absolute, "last_known": last_known}))
    lstm_all_y_true.append(y_test); lstm_all_pred.append(pred_absolute); lstm_all_last_known.append(last_known)
    print(f"  horizon {horizon_i}/10 (best val_residual_MAE={best_val_mae:.3f}): "
          f"LSTM MAE={lstm_mae:.3f} RMSE={lstm_rmse:.3f} R2={lstm_r2:.3f}  "
          f"|  Translation MAE={trans_mae:.3f} RMSE={trans_rmse:.3f} R2={trans_r2:.3f}")

    lstm_metrics_rows.append({
    "horizon": horizon_i,
    "LSTM": lstm_mae})

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

pl.concat(lstm_prediction_rows).write_parquet(f"{OUTPUT_DIR}/predictions_lstm.parquet")
print("Training complete.")

lstm_all_y_true, lstm_all_pred, lstm_all_last_known = np.concatenate(lstm_all_y_true), np.concatenate(lstm_all_pred), np.concatenate(lstm_all_last_known)
print(f"LSTM pooled:        MAE={mean_absolute_error(lstm_all_y_true, lstm_all_pred):.3f} "
      f"RMSE={np.sqrt(mean_squared_error(lstm_all_y_true, lstm_all_pred)):.3f} R2={r2_score(lstm_all_y_true, lstm_all_pred):.3f} (n={len(lstm_all_y_true):,})")
print(f"Translation pooled: MAE={mean_absolute_error(lstm_all_y_true, lstm_all_last_known):.3f} "
      f"RMSE={np.sqrt(mean_squared_error(lstm_all_y_true, lstm_all_last_known)):.3f} R2={r2_score(lstm_all_y_true, lstm_all_last_known):.3f}")

Training one LSTM regressor per horizon...
  horizon 1/10 (best val_residual_MAE=1.850): LSTM MAE=3.649 RMSE=12.119 R2=0.492  |  Translation MAE=3.844 RMSE=13.734 R2=0.348
  horizon 2/10 (best val_residual_MAE=1.530): LSTM MAE=3.258 RMSE=11.229 R2=0.465  |  Translation MAE=3.656 RMSE=12.948 R2=0.289
  horizon 3/10 (best val_residual_MAE=1.003): LSTM MAE=1.943 RMSE=12.809 R2=-0.137  |  Translation MAE=2.577 RMSE=10.146 R2=0.286
  horizon 4/10 (best val_residual_MAE=0.691): LSTM MAE=0.885 RMSE=5.114 R2=0.509  |  Translation MAE=1.511 RMSE=6.175 R2=0.285
  horizon 5/10 (best val_residual_MAE=0.422): LSTM MAE=0.541 RMSE=3.237 R2=0.669  |  Translation MAE=1.277 RMSE=4.361 R2=0.400
  horizon 6/10 (best val_residual_MAE=0.414): LSTM MAE=0.469 RMSE=3.472 R2=0.487  |  Translation MAE=1.204 RMSE=3.424 R2=0.501
  horizon 7/10 (best val_residual_MAE=0.360): LSTM MAE=0.413 RMSE=2.366 R2=0.742  |  Translation MAE=1.249 RMSE=3.438 R2=0.455
  horizon 8/10 (best val_residual_MAE=0.372): LSTM MAE=0.438 

# GNN

## Build the station graph and node features

Built once, from the raw training table, shared by feature selection, tuning, and production training below.

In [27]:
DATA_DIR = "/kaggle/input/datasets/ranjithpanicker/railway-data"
TRAIN_PATH = f"{DATA_DIR}/train_3city_2025-11_to_2026-04_final.parquet"
N_PAST_HISTORY = 3
INFRA_NUMERIC_VARS = ["gleisanzahl", "geschwindigkeit"]
MAX_PLAUSIBLE_DELAY_PREDICTION = 360
RESIDUAL_OUTPUT_BOUND = 150
GRAD_CLIP_MAX_NORM = 5.0

train_raw = pl.read_parquet(TRAIN_PATH)
pair_dfs = [train_raw.select([pl.col("current_station").alias("a"), pl.col("future_station_1").alias("b")]).drop_nulls().unique()]
for i in range(1, N_FUTURE):
    a_col, b_col = f"future_station_{i}", f"future_station_{i+1}"
    if a_col in train_raw.columns and b_col in train_raw.columns:
        pair_dfs.append(train_raw.select([pl.col(a_col).alias("a"), pl.col(b_col).alias("b")]).drop_nulls().unique())
all_pairs = pl.concat(pair_dfs, how="vertical_relaxed").unique()
stations = sorted(set(all_pairs["a"].to_list()) | set(all_pairs["b"].to_list()))
station_to_idx = {s: i for i, s in enumerate(stations)}
station_to_idx["__UNKNOWN_STATION__"] = len(station_to_idx)
unknown_idx = station_to_idx["__UNKNOWN_STATION__"]

edge_pairs = set()
for a, b in zip(all_pairs["a"].to_list(), all_pairs["b"].to_list()):
    ia, ib = station_to_idx[a], station_to_idx[b]
    edge_pairs.add((ia, ib)); edge_pairs.add((ib, ia))
n_stations = len(station_to_idx)

adjacency = torch.zeros((n_stations, n_stations), dtype=torch.float32)
for i, j in edge_pairs:
    adjacency[i, j] = 1.0
adjacency += torch.eye(n_stations)
degree = adjacency.sum(dim=1)
degree_inv_sqrt = torch.pow(degree.clamp(min=1e-8), -0.5)
norm_adj = (degree_inv_sqrt.unsqueeze(1) * adjacency * degree_inv_sqrt.unsqueeze(0)).to(DEVICE)

elek_categories = ["Oberleitung", "Stromschiene", "nicht elektrifiziert", "UNKNOWN"]
elek_to_idx = {c: i for i, c in enumerate(elek_categories)}
numeric_sums = np.zeros((n_stations, len(INFRA_NUMERIC_VARS)), dtype=np.float64)
numeric_counts = np.zeros(n_stations, dtype=np.float64)
elek_counts = np.zeros((n_stations, len(elek_categories)), dtype=np.float64)

for i in range(1, N_FUTURE + 1):
    station_col = f"future_station_{i}"
    if station_col not in train_raw.columns:
        continue
    select_cols = [station_col] + [f"future_{v}_{i}" for v in INFRA_NUMERIC_VARS if f"future_{v}_{i}" in train_raw.columns]
    elek_col = f"future_elektrifizierung_{i}"
    has_elek_col = elek_col in train_raw.columns
    if has_elek_col:
        select_cols.append(elek_col)
    for row in train_raw.select(select_cols).drop_nulls(subset=[station_col]).unique().iter_rows(named=True):
        station = row[station_col]
        if station not in station_to_idx:
            continue
        idx = station_to_idx[station]
        for v in INFRA_NUMERIC_VARS:
            val = row.get(f"future_{v}_{i}")
            if val is not None:
                numeric_sums[idx, INFRA_NUMERIC_VARS.index(v)] += val
        numeric_counts[idx] += 1
        if has_elek_col:
            elek_counts[idx, elek_to_idx.get(row.get(elek_col), elek_to_idx["UNKNOWN"])] += 1
            
numeric_avg = np.divide(numeric_sums, numeric_counts[:, None], out=np.zeros_like(numeric_sums), where=numeric_counts[:, None] > 0)
elek_onehot = np.zeros((n_stations, len(elek_categories)), dtype=np.float32)

for idx in range(n_stations):
    elek_onehot[idx, np.argmax(elek_counts[idx]) if elek_counts[idx].sum() > 0 else elek_to_idx["UNKNOWN"]] = 1.0
station_delay_pairs = [train_raw.select([pl.col(f"future_station_{i}").alias("station"), pl.col(f"future_delay_{i}").alias("delay")]).drop_nulls()
                       for i in range(1, N_FUTURE + 1) if f"future_station_{i}" in train_raw.columns and f"future_delay_{i}" in train_raw.columns]

pooled_delays = pl.concat(station_delay_pairs, how="vertical_relaxed")
lookup_grouped = pooled_delays.group_by("station").agg(pl.col("delay").mean())
station_avg_lookup = dict(zip(lookup_grouped["station"].to_list(), lookup_grouped["delay"].to_list()))
global_avg_delay = float(pooled_delays["delay"].mean())
avg_delay_per_station = np.array([station_avg_lookup.get(s, global_avg_delay) for s in station_to_idx.keys()], dtype=np.float32)
node_features = torch.tensor(np.concatenate([numeric_avg.astype(np.float32), elek_onehot, avg_delay_per_station[:, None]], axis=1),
                              dtype=torch.float32, device=DEVICE)

print(f"{n_stations} stations, node feature matrix: {tuple(node_features.shape)}")
del train_raw
import gc
gc.collect()

184 stations, node feature matrix: (184, 7)


93

## Feature-group to column mapping and model


In [28]:
GNN_GROUP_COLUMNS = {
    "infra_numeric": ["gleisanzahl", "geschwindigkeit"],
    "timetable": ["station_headway_min", "station_dwell_min", "station_freq_per_day"],
    "congestion_ahead": ["station_congestion_n", "station_congestion_avgdelay"],
    "past_minutes_ago": ["past_minutes_ago_1", "past_minutes_ago_2", "past_minutes_ago_3"],
    "cyclical_extended": ["hour_sin_2", "hour_sin_4", "hour_cos_2", "hour_cos_4",
                          "doy_sin_1", "doy_sin_2", "doy_sin_4", "doy_cos_1", "doy_cos_2", "doy_cos_4"]
                         + [f"dow_onehot_{d}" for d in range(7)],
}
GNN_STATIC_BASE_COLUMNS = ["hour_sin_1", "hour_cos_1", "dow_sin", "dow_cos",
                           "n_trains_at_current_station", "avg_delay_others_at_current_station",
                           "last_known_delay", "past_delay_1", "past_delay_2", "past_delay_3", "minutes_ahead"]

def gnn_static_columns(exclude_groups):
    return resolve_columns(GNN_STATIC_BASE_COLUMNS, GNN_GROUP_COLUMNS, exclude_groups)

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, norm_adj):
        return F.relu(norm_adj @ self.linear(x))

class StationEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.gc1 = GCNLayer(in_dim, hidden_dim)
        self.gc2 = GCNLayer(hidden_dim, out_dim)

    def forward(self, node_features, norm_adj):
        return self.gc2(self.gc1(node_features, norm_adj), norm_adj)

class MLPStationEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, node_features, norm_adj=None):
        return F.relu(self.fc2(F.relu(self.fc1(node_features))))

class DelayGNN(nn.Module):
    def __init__(self, n_static_numeric, n_train_types, n_elektrifizierung, node_feat_dim,
                 gnn_hidden, gnn_out, hidden_size, n_layers, dropout, output_bound=RESIDUAL_OUTPUT_BOUND,
                 station_encoder_cls=StationEncoder):
        super().__init__()
        self.station_encoder = station_encoder_cls(node_feat_dim, gnn_hidden, gnn_out)
        self.train_type_embed = nn.Embedding(n_train_types, EMBED_DIM)
        self.has_elektrifizierung = n_elektrifizierung > 0
        if self.has_elektrifizierung:
            self.elek_embed = nn.Embedding(n_elektrifizierung, EMBED_DIM)
        self.output_bound = output_bound
        input_dim = n_static_numeric + gnn_out + EMBED_DIM + (EMBED_DIM if self.has_elektrifizierung else 0)
        layers = []
        d = input_dim
        for _ in range(n_layers):
            layers += [nn.Linear(d, hidden_size), nn.ReLU(), nn.Dropout(dropout)]
            d = hidden_size
        layers += [nn.Linear(d, 1)]
        self.head = nn.Sequential(*layers)

    def forward(self, static_numeric, train_type_ids, current_station_idx, node_features, norm_adj, elektrifizierung_ids=None):
        station_embeds_all = self.station_encoder(node_features, norm_adj)
        station_embed = station_embeds_all[current_station_idx]
        parts = [static_numeric, station_embed, self.train_type_embed(train_type_ids)]
        if self.has_elektrifizierung and elektrifizierung_ids is not None:
            parts.append(self.elek_embed(elektrifizierung_ids))
        raw = self.head(torch.cat(parts, dim=1)).squeeze(-1)
        return torch.tanh(raw / self.output_bound) * self.output_bound

def gnn_batched_forward(model, batch_size, static_numeric, tt, cs, node_features, norm_adj, elek=None):
    preds = []
    n = static_numeric.shape[0]
    for start in range(0, n, batch_size):
        end = start + batch_size
        preds.append(model(static_numeric[start:end], tt[start:end], cs[start:end], node_features, norm_adj,
                            elek[start:end] if elek is not None else None))
    return torch.cat(preds, dim=0)

def gnn_train_one_model(train_df, static_cols, has_elek, n_train_types, n_elektrifizierung,
                         hidden_size, n_layers, dropout, learning_rate, gnn_hidden_dim, gnn_out_dim,
                         max_epochs, patience, station_encoder_cls=StationEncoder):
    fit_idx, val_idx = temporal_split_indices(train_df["snapshot_time"].to_numpy())
    fit_df, val_df = train_df[fit_idx], train_df[val_idx]
    static_scaler = StandardScaler().fit(fit_df.select(static_cols).to_numpy())

    def make_tensors(df):
        static = static_scaler.transform(df.select(static_cols).to_numpy())
        tt = df["train_type_enc"].to_numpy()
        elek = df["elektrifizierung_enc"].to_numpy() if has_elek else None
        cs = np.array([station_to_idx.get(s, unknown_idx) for s in df["current_station"].to_list()], dtype=np.int64)
        y = (df["target"] - df["last_known_delay"]).to_numpy()
        return (torch.tensor(static, dtype=torch.float32, device=DEVICE),
                torch.tensor(tt, dtype=torch.long, device=DEVICE),
                torch.tensor(cs, dtype=torch.long, device=DEVICE),
                torch.tensor(elek, dtype=torch.long, device=DEVICE) if has_elek else None,
                torch.tensor(y, dtype=torch.float32, device=DEVICE))

    static_fit, tt_fit, cs_fit, elek_fit, y_fit = make_tensors(fit_df)
    static_val, tt_val, cs_val, elek_val, y_val = make_tensors(val_df)

    torch.manual_seed(RANDOM_STATE)
    model = DelayGNN(len(static_cols), n_train_types, n_elektrifizierung, node_features.shape[1],
                      gnn_hidden_dim, gnn_out_dim, hidden_size, n_layers, dropout,
                      station_encoder_cls=station_encoder_cls).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=LR_SCHEDULER_FACTOR,
                                                            patience=LR_SCHEDULER_PATIENCE)
    loss_fn = nn.L1Loss()
    n_fit = static_fit.shape[0]
    best_val_mae, best_state, epochs_without_improve = float("inf"), None, 0
    for epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(n_fit, device=DEVICE)
        for start in range(0, n_fit, BATCH_SIZE):
            batch_idx = perm[start:start + BATCH_SIZE]
            optimizer.zero_grad()
            preds = model(static_fit[batch_idx], tt_fit[batch_idx], cs_fit[batch_idx], node_features, norm_adj,
                          elek_fit[batch_idx] if elek_fit is not None else None)
            loss = loss_fn(preds, y_fit[batch_idx])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_MAX_NORM)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_mae = mean_absolute_error(y_val.cpu().numpy(),
                                          gnn_batched_forward(model, BATCH_SIZE, static_val, tt_val, cs_val, node_features, norm_adj, elek_val).cpu().numpy())
        scheduler.step(val_mae)
        if val_mae < best_val_mae - 1e-4:
            best_val_mae, epochs_without_improve, best_state = val_mae, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    del static_fit, tt_fit, cs_fit, elek_fit, y_fit, static_val, tt_val, cs_val, elek_val, y_val
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return model, best_val_mae, static_scaler

## Feature selection: greedy backward elimination

Runs the greedy backward elimination from the Helper functions section against this model's own validation loop, at the near(+2) / mid(+5) / far(+8) representative horizons.


In [29]:
GNN_FIXED_HP = {"hidden_size": 64, "n_layers": 2, "dropout": 0.1,
                "learning_rate": 1e-2, "gnn_hidden_dim": 8, "gnn_out_dim": 4}

GNN_SELECTION_MAX_EPOCHS, GNN_SELECTION_PATIENCE = 15, 5

def gnn_evaluate_feature_config(horizon_i, exclude_groups):
    train_df = pl.read_parquet(f"{FEATURES_DIR}/train_h{horizon_i}.parquet")
    static_cols = gnn_static_columns(exclude_groups)
    has_elek = "elektrifizierung" not in exclude_groups
    n_train_types = train_df["train_type_enc"].max() + 1
    n_elektrifizierung = (train_df["elektrifizierung_enc"].max() + 1) if has_elek else 0
    _, val_mae, _ = gnn_train_one_model(train_df, static_cols, has_elek, n_train_types, n_elektrifizierung,
                                        GNN_FIXED_HP["hidden_size"], GNN_FIXED_HP["n_layers"], GNN_FIXED_HP["dropout"],
                                        GNN_FIXED_HP["learning_rate"], GNN_FIXED_HP["gnn_hidden_dim"], GNN_FIXED_HP["gnn_out_dim"],
                                        GNN_SELECTION_MAX_EPOCHS, GNN_SELECTION_PATIENCE)
    return val_mae

print(f"GNN feature selection (candidate groups: {ALL_CANDIDATE_GROUPS})...")
run_greedy_selection(gnn_evaluate_feature_config, metric_label="val residual MAE")
print("\nFeature selection complete.")

GNN feature selection (candidate groups: ['weather', 'infra_numeric', 'elektrifizierung', 'timetable', 'congestion_ahead', 'cyclical_extended', 'past_minutes_ago'])...

--- band near (horizon 2): baseline val residual MAE=2.2088 ---
  remove 'cyclical_extended': val residual MAE=1.7172 (within tolerance, removed)
  remove 'congestion_ahead': val residual MAE=1.6968 (within tolerance, removed)
  remove 'elektrifizierung': val residual MAE=1.6332 (within tolerance, removed)
  remove 'weather': val residual MAE=1.6332 (within tolerance, removed)
  stop: best remaining candidate ('infra_numeric', val residual MAE=1.7786) would meaningfully hurt performance
  final for near: kept=['infra_numeric', 'timetable', 'past_minutes_ago'], excluded=['cyclical_extended', 'congestion_ahead', 'elektrifizierung', 'weather']

--- band mid (horizon 5): baseline val residual MAE=0.7314 ---
  remove 'cyclical_extended': val residual MAE=0.5963 (within tolerance, removed)
  remove 'weather': val residual MAE

## Feature Selection Config

The result of feature selection is saved here for the next stages, which includes tuning the model and final training.

`model_EXCLUDE_FEATURE_GROUPS` stores the features that gets excluded for all the horizon models (10 models).

`model_PERMANENT_HORIZON_FEATURE_OVERRIDES` stores the features that are corresponding to horizon level feature exclusion (single category of model i.e near/medium/far), these features are excluded only for the specified horizon, they are available for other horizons to train on.

In [29]:
GNN_EXCLUDE_FEATURE_GROUPS = ["cyclical_extended", "weather"]

GNN_PERMANENT_HORIZON_FEATURE_OVERRIDES = {
    1: ["congestion_ahead", "elektrifizierung"], 2: ["congestion_ahead", "elektrifizierung"], 3: ["congestion_ahead", "elektrifizierung"],}

## Hyperparameter tuning: Optuna

Tuning here is run against each tuning horizon's own confirmed feature exclusions, the same pattern LSTM's tuning uses.

In [65]:
GNN_N_TUNING_TRIALS = 15
GNN_TUNING_MAX_EPOCHS, GNN_TUNING_PATIENCE = 30, 8


print(f"GNN hyperparameter tuning ({GNN_N_TUNING_TRIALS} trials, pooled across horizons {TUNING_HORIZONS})...")
gnn_tuning_dfs = {h: pl.read_parquet(f"{FEATURES_DIR}/train_h{h}.parquet") for h in TUNING_HORIZONS}

def gnn_objective(trial):
    hp = {
        "hidden_size": trial.suggest_categorical("hidden_size", [32, 64, 128, 256]),
        "n_layers": trial.suggest_int("n_layers", 1, 5),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "gnn_hidden_dim": trial.suggest_categorical("gnn_hidden_dim", [8, 16, 32]),
        "gnn_out_dim": trial.suggest_categorical("gnn_out_dim", [4, 8, 16]),
    }
    val_maes = []
    for h, train_df in gnn_tuning_dfs.items():
        exclude_groups = resolve_exclude_groups(GNN_EXCLUDE_FEATURE_GROUPS, GNN_PERMANENT_HORIZON_FEATURE_OVERRIDES, h)
        static_cols = gnn_static_columns(exclude_groups)
        has_elek = "elektrifizierung" not in exclude_groups
        n_train_types = train_df["train_type_enc"].max() + 1
        n_elektrifizierung = (train_df["elektrifizierung_enc"].max() + 1) if has_elek else 0
        _, val_mae, _ = gnn_train_one_model(train_df, static_cols, has_elek, n_train_types, n_elektrifizierung,
                                            hp["hidden_size"], hp["n_layers"], hp["dropout"], hp["learning_rate"],
                                            hp["gnn_hidden_dim"], hp["gnn_out_dim"], GNN_TUNING_MAX_EPOCHS, GNN_TUNING_PATIENCE)
        val_maes.append(val_mae)
    return float(np.mean(val_maes))

gnn_study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=5))
gnn_study.optimize(gnn_objective, n_trials=GNN_N_TUNING_TRIALS, show_progress_bar=False)
print(f"Best trial: val_residual_MAE={gnn_study.best_value:.3f}")
print(f"Best params: {gnn_study.best_params}")
print("Hyperparameter tuning complete.")

GNN hyperparameter tuning (15 trials, pooled across horizons [2, 5, 8])...


[I 2026-09-23 18:55:34,575] A new study created in memory with name: no-name-7e7dfcc3-b7bb-4400-9ce8-1e2d63c4d8ad
[I 2026-09-23 18:57:29,053] Trial 0 finished with value: 1.2886352340380351 and parameters: {'hidden_size': 64, 'n_layers': 1, 'dropout': 0.07799726016810132, 'learning_rate': 0.00013066739238053285, 'gnn_hidden_dim': 8, 'gnn_out_dim': 8}. Best is trial 0 with value: 1.2886352340380351.
[I 2026-09-23 18:59:54,081] Trial 1 finished with value: 0.822856068611145 and parameters: {'hidden_size': 256, 'n_layers': 3, 'dropout': 0.21597250932105788, 'learning_rate': 0.0003823475224675188, 'gnn_hidden_dim': 8, 'gnn_out_dim': 16}. Best is trial 1 with value: 0.822856068611145.
[I 2026-09-23 19:02:32,798] Trial 2 finished with value: 0.9632948239644369 and parameters: {'hidden_size': 128, 'n_layers': 4, 'dropout': 0.08526206184364576, 'learning_rate': 0.00013492834268013249, 'gnn_hidden_dim': 16, 'gnn_out_dim': 16}. Best is trial 1 with value: 0.822856068611145.
[I 2026-09-23 19:05:2

Best trial: val_residual_MAE=0.624
Best params: {'hidden_size': 256, 'n_layers': 3, 'dropout': 0.18439996575247108,'learning_rate': 0.0011364695031944965, 'gnn_hidden_dim': 32, 'gnn_out_dim': 8}
Hyperparameter tuning complete.


## Confirmed configuration


In [31]:
GNN_PRODUCTION_HP = {"hidden_size": 256, "n_layers": 3, "dropout": 0.18439996575247108,
                      "learning_rate": 0.0011364695031944965, "gnn_hidden_dim": 32, "gnn_out_dim": 8}

GNN_MAX_EPOCHS, GNN_PATIENCE = 120, 15

save_model_config("gnn", {
    "exclude_feature_groups": GNN_EXCLUDE_FEATURE_GROUPS,
    "permanent_horizon_feature_overrides": {str(k): v for k, v in GNN_PERMANENT_HORIZON_FEATURE_OVERRIDES.items()},
    "production_hp": GNN_PRODUCTION_HP,
})

Saved confirmed configuration to /kaggle/working/gnn_config.json


## Train and evaluate every horizon

Trains one model per horizon (1 to 10) on the full training data, using the confirmed configuration above, and evaluates each once on the held-out test set.


In [32]:
print("Training one GNN regressor per horizon...")
gnn_prediction_rows, gnn_all_y_true, gnn_all_pred, gnn_all_last_known = [], [], [], []
gnn_metrics_rows = []

for horizon_i in range(1, N_FUTURE + 1):
    train_df = pl.read_parquet(f"{FEATURES_DIR}/train_h{horizon_i}.parquet")
    test_df = pl.read_parquet(f"{FEATURES_DIR}/test_h{horizon_i}.parquet")
    exclude_groups = resolve_exclude_groups(GNN_EXCLUDE_FEATURE_GROUPS, GNN_PERMANENT_HORIZON_FEATURE_OVERRIDES, horizon_i)
    static_cols = gnn_static_columns(exclude_groups)
    has_elek = "elektrifizierung" not in exclude_groups
    n_train_types = train_df["train_type_enc"].max() + 1
    n_elektrifizierung = (train_df["elektrifizierung_enc"].max() + 1) if has_elek else 0

    model, best_val_mae, static_scaler = gnn_train_one_model(
        train_df, static_cols, has_elek, n_train_types, n_elektrifizierung,
        GNN_PRODUCTION_HP["hidden_size"], GNN_PRODUCTION_HP["n_layers"], GNN_PRODUCTION_HP["dropout"],
        GNN_PRODUCTION_HP["learning_rate"], GNN_PRODUCTION_HP["gnn_hidden_dim"], GNN_PRODUCTION_HP["gnn_out_dim"],
        GNN_MAX_EPOCHS, GNN_PATIENCE)

    static_test = torch.tensor(static_scaler.transform(test_df.select(static_cols).to_numpy()), dtype=torch.float32, device=DEVICE)
    tt_test = torch.tensor(test_df["train_type_enc"].to_numpy(), dtype=torch.long, device=DEVICE)
    elek_test = torch.tensor(test_df["elektrifizierung_enc"].to_numpy(), dtype=torch.long, device=DEVICE) if has_elek else None
    cs_test = torch.tensor([station_to_idx.get(s, unknown_idx) for s in test_df["current_station"].to_list()], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        pred_residual = gnn_batched_forward(model, BATCH_SIZE, static_test, tt_test, cs_test, node_features, norm_adj, elek_test).cpu().numpy()
    last_known = test_df["last_known_delay"].to_numpy()
    y_test = test_df["target"].to_numpy()
    pred_absolute = np.clip(pred_residual + last_known, -MAX_PLAUSIBLE_DELAY_PREDICTION, MAX_PLAUSIBLE_DELAY_PREDICTION)

    gnn_mae, gnn_rmse, gnn_r2 = mean_absolute_error(y_test, pred_absolute), np.sqrt(mean_squared_error(y_test, pred_absolute)), r2_score(y_test, pred_absolute)
    trans_mae, trans_rmse, trans_r2 = mean_absolute_error(y_test, last_known), np.sqrt(mean_squared_error(y_test, last_known)), r2_score(y_test, last_known)
    gnn_prediction_rows.append(pl.DataFrame({"horizon": horizon_i, "y_true": y_test, "pred": pred_absolute, "last_known": last_known}))
    gnn_all_y_true.append(y_test); gnn_all_pred.append(pred_absolute); gnn_all_last_known.append(last_known)
    print(f"  horizon {horizon_i}/10 (best val_residual_MAE={best_val_mae:.3f}): "
          f"GNN MAE={gnn_mae:.3f} RMSE={gnn_rmse:.3f} R2={gnn_r2:.3f}  "
          f"|  Translation MAE={trans_mae:.3f} RMSE={trans_rmse:.3f} R2={trans_r2:.3f}")
    
    gnn_metrics_rows.append({
    "horizon": horizon_i,
    "GNN": gnn_mae})

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

pl.concat(gnn_prediction_rows).write_parquet(f"{OUTPUT_DIR}/predictions_gnn.parquet")
print("Training complete.")

gnn_all_y_true, gnn_all_pred, gnn_all_last_known = np.concatenate(gnn_all_y_true), np.concatenate(gnn_all_pred), np.concatenate(gnn_all_last_known)
print(f"GNN pooled:         MAE={mean_absolute_error(gnn_all_y_true, gnn_all_pred):.3f} "
      f"RMSE={np.sqrt(mean_squared_error(gnn_all_y_true, gnn_all_pred)):.3f} R2={r2_score(gnn_all_y_true, gnn_all_pred):.3f} (n={len(gnn_all_y_true):,})")
print(f"Translation pooled: MAE={mean_absolute_error(gnn_all_y_true, gnn_all_last_known):.3f} "
      f"RMSE={np.sqrt(mean_squared_error(gnn_all_y_true, gnn_all_last_known)):.3f} R2={r2_score(gnn_all_y_true, gnn_all_last_known):.3f}")

Training one GNN regressor per horizon...
  horizon 1/10 (best val_residual_MAE=1.715): GNN MAE=3.327 RMSE=11.048 R2=0.578  |  Translation MAE=3.844 RMSE=13.734 R2=0.348
  horizon 2/10 (best val_residual_MAE=1.398): GNN MAE=3.046 RMSE=10.564 R2=0.527  |  Translation MAE=3.656 RMSE=12.948 R2=0.289
  horizon 3/10 (best val_residual_MAE=0.886): GNN MAE=1.654 RMSE=8.000 R2=0.556  |  Translation MAE=2.577 RMSE=10.146 R2=0.286
  horizon 4/10 (best val_residual_MAE=0.528): GNN MAE=0.697 RMSE=4.536 R2=0.614  |  Translation MAE=1.511 RMSE=6.175 R2=0.285
  horizon 5/10 (best val_residual_MAE=0.320): GNN MAE=0.424 RMSE=3.154 R2=0.686  |  Translation MAE=1.277 RMSE=4.361 R2=0.400
  horizon 6/10 (best val_residual_MAE=0.299): GNN MAE=0.336 RMSE=1.591 R2=0.892  |  Translation MAE=1.204 RMSE=3.424 R2=0.501
  horizon 7/10 (best val_residual_MAE=0.285): GNN MAE=0.329 RMSE=1.729 R2=0.862  |  Translation MAE=1.249 RMSE=3.438 R2=0.455
  horizon 8/10 (best val_residual_MAE=0.309): GNN MAE=0.353 RMSE=1.598 

### Model results

Visualizing the MAE scores for each model against the translation baseline


In [60]:
mae = (pd.concat([pd.DataFrame(metrics_rows).rename(columns={'xgboost': 'XGBoost', 'translation': 'Translation'}).set_index("horizon"),
                 pd.DataFrame(lstm_metrics_rows).set_index("horizon"),
                 pd.DataFrame(mpl_metrics_rows).set_index("horizon"),
                 pd.DataFrame(gnn_metrics_rows).rename(columns={"LSTM":'GNN'}).set_index("horizon")],axis=1).reset_index())

mae

,horizon,XGBoost,Translation,LSTM,MLP,GNN
0,1,3.808605,3.843985,3.649122,3.464355,3.326615
1,2,3.241580,3.655651,3.257988,2.950454,3.046147
2,3,1.982121,2.577229,1.943492,1.858104,1.653918
3,4,0.864034,1.511087,0.885296,0.758948,0.696582
4,5,0.629996,1.276765,0.540748,0.445503,0.423709
5,6,0.472832,1.204485,0.468533,0.352931,0.336411
6,7,0.451479,1.249294,0.412705,0.330268,0.328953
7,8,0.435936,1.293754,0.438283,0.335698,0.353154
8,9,0.453485,1.347278,0.439365,0.338233,0.355434
9,10,0.454952,1.406642,0.454281,0.380301,0.367540


In [63]:
pio.renderers.default = "colab"

horizons = mae["horizon"]
COLORS = {"Translation": "#7f7f7f", "XGBoost": "#1f77b4", "MLP": "#ff7f0e", "LSTM": "#2ca02c", "GNN": "#d62728"}

LAYOUT_KWARGS = dict(template="plotly_white", font=dict(family="Arial, sans-serif", size=14, color="#222222"),
                     legend=dict(orientation="h", yanchor="bottom", y=-0.28, xanchor="center", x=0.5, title=None, font=dict(size=13)),margin=dict(l=70, r=30, t=60, b=90),
                     width=850, height=520,)

AXIS_KWARGS = dict(showgrid=True, gridcolor="#e6e6e6", zeroline=False, showline=True, linecolor="#999999")

fig1 = go.Figure()

for name in ["Translation", "XGBoost", "MLP", "LSTM", "GNN"]:
    is_baseline = name == "Translation"
    fig1.add_trace(go.Scatter(x=mae["horizon"],y=mae[name], name=name, mode="lines+markers", line=dict(color=COLORS[name], width=2.4, dash="dash" if is_baseline else "solid"),
                              marker=dict(size=7 if not is_baseline else 0),))

fig1.update_layout(
    title=dict(text="Test-set MAE by prediction horizon", x=0.5, xanchor="center", font=dict(size=18)),
    xaxis=dict(title="Prediction horizon (stops ahead)", tickmode="linear", dtick=1, **AXIS_KWARGS),
    yaxis=dict(title="MAE (minutes)", **AXIS_KWARGS),
    **LAYOUT_KWARGS,)
fig1.show()


fig2 = go.Figure()

for name in ["XGBoost", "MLP", "LSTM", "GNN"]:
    pct = (mae["Translation"] - mae[name]) / mae["Translation"] * 100
    fig2.add_trace(go.Scatter(x=mae["horizon"], y=pct, name=name, mode="lines+markers", line=dict(color=COLORS[name], width=2.4), marker=dict(size=7),))

fig2.add_hline(y=0, line_color="#999999", line_width=1)

fig2.update_layout(
    title=dict(text="Model advantage over baseline, by horizon", x=0.5, xanchor="center", font=dict(size=18)),
    xaxis=dict(title="Prediction horizon (stops ahead)", tickmode="linear", dtick=1, **AXIS_KWARGS),
    yaxis=dict(title="MAE improvement over translation baseline (%)", ticksuffix="%", **AXIS_KWARGS),
    **LAYOUT_KWARGS,)
fig2.show()

## Topology Ablation

Reruns the confirmed GNN with its graph-convolutional station encoder replaced by a parameter-matched, non-graph MLP encoder, everything else held fixed, to isolate whether the graph structure itself, not just model capacity, improves prediction (RQ4).


In [13]:
gnn_preds_df = pl.read_parquet(f"{OUTPUT_DIR}/predictions_gnn.parquet")

print("Topology ablation: GNN with a non-graph per-station MLP encoder, all ten horizons...")
topology_rows = []
topology_all_y_true, topology_all_pred = [], []

for horizon_i in range(1, N_FUTURE + 1):
    train_df = pl.read_parquet(f"{FEATURES_DIR}/train_h{horizon_i}.parquet")
    test_df = pl.read_parquet(f"{FEATURES_DIR}/test_h{horizon_i}.parquet")
    exclude_groups = resolve_exclude_groups(GNN_EXCLUDE_FEATURE_GROUPS, GNN_PERMANENT_HORIZON_FEATURE_OVERRIDES, horizon_i)
    static_cols = gnn_static_columns(exclude_groups)
    has_elek = "elektrifizierung" not in exclude_groups
    n_train_types = train_df["train_type_enc"].max() + 1
    n_elektrifizierung = (train_df["elektrifizierung_enc"].max() + 1) if has_elek else 0

    model, best_val_mae, static_scaler = gnn_train_one_model(
        train_df, static_cols, has_elek, n_train_types, n_elektrifizierung,
        GNN_PRODUCTION_HP["hidden_size"], GNN_PRODUCTION_HP["n_layers"], GNN_PRODUCTION_HP["dropout"],
        GNN_PRODUCTION_HP["learning_rate"], GNN_PRODUCTION_HP["gnn_hidden_dim"], GNN_PRODUCTION_HP["gnn_out_dim"],
        GNN_MAX_EPOCHS, GNN_PATIENCE, station_encoder_cls=MLPStationEncoder)

    static_test = torch.tensor(static_scaler.transform(test_df.select(static_cols).to_numpy()), dtype=torch.float32, device=DEVICE)
    tt_test = torch.tensor(test_df["train_type_enc"].to_numpy(), dtype=torch.long, device=DEVICE)
    elek_test = torch.tensor(test_df["elektrifizierung_enc"].to_numpy(), dtype=torch.long, device=DEVICE) if has_elek else None
    cs_test = torch.tensor([station_to_idx.get(s, unknown_idx) for s in test_df["current_station"].to_list()], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        pred_residual = gnn_batched_forward(model, BATCH_SIZE, static_test, tt_test, cs_test, node_features, norm_adj, elek_test).cpu().numpy()
    last_known = test_df["last_known_delay"].to_numpy()
    y_test = test_df["target"].to_numpy()
    pred_absolute = np.clip(pred_residual + last_known, -MAX_PLAUSIBLE_DELAY_PREDICTION, MAX_PLAUSIBLE_DELAY_PREDICTION)

    topo_mae, topo_rmse, topo_r2 = mean_absolute_error(y_test, pred_absolute), np.sqrt(mean_squared_error(y_test, pred_absolute)), r2_score(y_test, pred_absolute)
    topology_rows.append(pl.DataFrame({"horizon": horizon_i, "y_true": y_test, "pred": pred_absolute, "last_known": last_known}))
    topology_all_y_true.append(y_test); topology_all_pred.append(pred_absolute)

    gnn_h = gnn_preds_df.filter(pl.col("horizon") == horizon_i)
    gnn_mae_h = mean_absolute_error(gnn_h["y_true"].to_numpy(), gnn_h["pred"].to_numpy())
    gnn_rmse_h = np.sqrt(mean_squared_error(gnn_h["y_true"].to_numpy(), gnn_h["pred"].to_numpy()))
    gnn_r2_h = r2_score(gnn_h["y_true"].to_numpy(), gnn_h["pred"].to_numpy())
    print(f"  horizon {horizon_i}/10 (best val_residual_MAE={best_val_mae:.3f}): "
          f"No-topology MAE={topo_mae:.3f} RMSE={topo_rmse:.3f} R2={topo_r2:.3f}  "
          f"|  GNN (with topology) MAE={gnn_mae_h:.3f} RMSE={gnn_rmse_h:.3f} R2={gnn_r2_h:.3f}")

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

pl.concat(topology_rows).write_parquet(f"{OUTPUT_DIR}/predictions_gnn_no_topology.parquet")
print("Topology ablation training complete.")

topology_all_y_true, topology_all_pred = np.concatenate(topology_all_y_true), np.concatenate(topology_all_pred)
no_topo_mae = mean_absolute_error(topology_all_y_true, topology_all_pred)
no_topo_rmse = np.sqrt(mean_squared_error(topology_all_y_true, topology_all_pred))
no_topo_r2 = r2_score(topology_all_y_true, topology_all_pred)

gnn_y_true_all, gnn_pred_all = gnn_preds_df["y_true"].to_numpy(), gnn_preds_df["pred"].to_numpy()
gnn_mae = mean_absolute_error(gnn_y_true_all, gnn_pred_all)
gnn_rmse = np.sqrt(mean_squared_error(gnn_y_true_all, gnn_pred_all))
gnn_r2 = r2_score(gnn_y_true_all, gnn_pred_all)

print(f"\nGNN (no topology) pooled:   MAE={no_topo_mae:.3f} RMSE={no_topo_rmse:.3f} R2={no_topo_r2:.3f} (n={len(topology_all_y_true):,})")
print(f"GNN (with topology) pooled: MAE={gnn_mae:.3f} RMSE={gnn_rmse:.3f} R2={gnn_r2:.3f}")
print(f"Difference (no-topology minus with-topology): "
      f"MAE={no_topo_mae - gnn_mae:+.3f}, RMSE={no_topo_rmse - gnn_rmse:+.3f}, R2={no_topo_r2 - gnn_r2:+.3f}")


Topology ablation: GNN with a non-graph per-station MLP encoder, all ten horizons...
  horizon 1/10 (best val_residual_MAE=1.698): No-topology MAE=3.416 RMSE=11.455 R2=0.546  |  GNN (with topology) MAE=3.327 RMSE=11.048 R2=0.578
  horizon 2/10 (best val_residual_MAE=1.325): No-topology MAE=2.886 RMSE=10.183 R2=0.560  |  GNN (with topology) MAE=3.046 RMSE=10.564 R2=0.527
  horizon 3/10 (best val_residual_MAE=0.878): No-topology MAE=1.639 RMSE=8.093 R2=0.546  |  GNN (with topology) MAE=1.654 RMSE=8.000 R2=0.556
  horizon 4/10 (best val_residual_MAE=0.561): No-topology MAE=0.748 RMSE=5.012 R2=0.529  |  GNN (with topology) MAE=0.697 RMSE=4.536 R2=0.614
  horizon 5/10 (best val_residual_MAE=0.312): No-topology MAE=0.408 RMSE=2.834 R2=0.747  |  GNN (with topology) MAE=0.424 RMSE=3.154 R2=0.686
  horizon 6/10 (best val_residual_MAE=0.299): No-topology MAE=0.325 RMSE=1.989 R2=0.832  |  GNN (with topology) MAE=0.336 RMSE=1.591 R2=0.892
  horizon 7/10 (best val_residual_MAE=0.302): No-topology M

# THE END